In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
def print_summary(cmask, rmask, field=None):
    if np.sum(rmask)==0:
        print('Zero area')
        return None
    if field is None:
        print('Total: {:.2f}% area; Density = {:.2f}'.format(100*np.sum(np.sum(rmask))/len(randoms), np.sum(cmask)/np.sum(rmask)*randoms_density))
    elif field=='south':
        print('South: {:.2f}% area; Density = {:.2f}'.format(100*np.sum(np.sum(rmask & rsouth))/np.sum(rsouth), np.sum(cmask & south)/np.sum(rmask & rsouth)*randoms_density))
    elif field=='north':
        print('North: {:.2f}% area; Density = {:.2f}'.format(100*np.sum(np.sum(rmask & rnorth))/np.sum(rnorth), np.sum(cmask & north)/np.sum(rmask & rnorth)*randoms_density))

In [4]:
# new_mask_dir = '/global/cfs/cdirs/desi/users/rongpu/desi_mask/dev'

# n_randoms_catalogs = 1
# randoms_columns = ['RA', 'DEC', 'NOBS_G', 'NOBS_R', 'NOBS_Z', 'MASKBITS', 'PHOTSYS', 'WISEMASK_W1']
# randoms_paths = sorted(glob.glob('/global/cfs/cdirs/desi/target/catalogs/dr9/0.49.0/randoms/resolve/randoms-[0-9]*.fits'))
# randoms_paths = randoms_paths[:n_randoms_catalogs]

# randoms_stack = []

# for randoms_path in randoms_paths:
#     wmask_path = os.path.join(new_mask_dir, os.path.basename(randoms_path).replace('.fits', '-wisemask.npz'))
#     gmask_path = os.path.join(new_mask_dir, os.path.basename(randoms_path).replace('.fits', '-gaiamask.npz'))

#     randoms = Table(fitsio.read(randoms_path, columns=randoms_columns))
#     wmask = np.load(wmask_path)
#     gmask = np.load(gmask_path)
    
#     randoms['wise_mask'] = wmask['wise_mask']
#     randoms['gaia_mask'] = gmask['gaia_mask']
#     randoms['gaia_bright_mask'] = gmask['gaia_bright_mask']
    
#     randoms_stack.append(randoms)

# randoms = vstack(randoms_stack)
# print(len(randoms))
# randoms.write('/global/cfs/cdirs/desi/users/rongpu/tmp/randoms_new_mask.fits', overwrite=True)

randoms_density = 2500.
randoms = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/tmp/randoms_new_mask.fits'))

# Apply the same mask bits and NOBS requirements as the LRG targets
min_nobs = 1
desi_maskbits = [1, 12, 13]
mask = (randoms['NOBS_G']>=min_nobs) & (randoms['NOBS_R']>=min_nobs) & (randoms['NOBS_Z']>=min_nobs)
print(np.sum(~mask)/len(randoms))
randoms = randoms[mask]
randoms_clean = np.ones(len(randoms), dtype=bool)
for bit in desi_maskbits:
    randoms_clean &= (randoms['MASKBITS'] & 2**bit)==0
print(np.sum(~randoms_clean)/len(randoms))
randoms = randoms[randoms_clean]
print(len(randoms))

0.04709026232939822
0.01030547684545959
48794148


In [5]:
cat_stack = []

for field in ['south', 'north']:
    cat = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/targets/dr9.0/1.0.0/resolve/dr9_lrg_{}_1.0.0_basic.fits'.format(field)))
    cat_wisemask = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/targets/dr9.0/1.0.0/resolve/dr9_lrg_{}_1.0.0_wisemask.fits'.format(field)))
    wmask = np.load('/global/cfs/cdirs/desi/users/rongpu/desi_mask/dev/dr9_lrg_{}_1.0.0_basic-wisemask.npz'.format(field))
    gmask = np.load('/global/cfs/cdirs/desi/users/rongpu/desi_mask/dev/dr9_lrg_{}_1.0.0_basic-gaiamask.npz'.format(field))

    cat = hstack([cat, cat_wisemask], join_type='exact')
    cat['wise_mask'] = wmask['wise_mask']
    cat['gaia_mask'] = gmask['gaia_mask']
    cat['gaia_bright_mask'] = gmask['gaia_bright_mask']
    
    cat_stack.append(cat)
    
cat = vstack(cat_stack)
print(len(cat))

12338990


In [6]:
# north = cat['PHOTSYS']=='N'
# south = (cat['PHOTSYS']=='S') & (cat['DEC']>-15)
# rnorth = randoms['PHOTSYS']=='N'
# rsouth = (randoms['PHOTSYS']=='S') & (randoms['DEC']>-15)

north = cat['PHOTSYS']=='N'
south = cat['PHOTSYS']=='S'
rnorth = randoms['PHOTSYS']=='N'
rsouth = randoms['PHOTSYS']=='S'

In [7]:
# No masking
print_summary(np.full(len(cat), True), np.full(len(randoms), True))
print_summary(np.full(len(cat), True), np.full(len(randoms), True), field='south')
print_summary(np.full(len(cat), True), np.full(len(randoms), True), field='north')

Total: 100.00% area; Density = 632.20
South: 100.00% area; Density = 630.39
North: 100.00% area; Density = 637.42


In [8]:
# Masking using DR9 MASKBITS

dr9_maskbits = [1, 8, 11, 12, 13]

dr9_mask = np.zeros(len(cat), dtype=bool)
rdr9_mask = np.zeros(len(randoms), dtype=bool)
for bit in dr9_maskbits:
    dr9_mask |= (cat['MASKBITS'] & 2**bit)>0
    rdr9_mask |= (randoms['MASKBITS'] & 2**bit)>0
    
dr9_clean = ~dr9_mask
rdr9_clean = ~rdr9_mask

# Unmasked objects
print_summary(dr9_clean, rdr9_clean)
print_summary(dr9_clean, rdr9_clean, field='south')
print_summary(dr9_clean, rdr9_clean, field='north')
print()

# Masked objects
print_summary(dr9_mask, rdr9_mask)
print_summary(dr9_mask, rdr9_mask, field='south')
print_summary(dr9_mask, rdr9_mask, field='north')

Total: 93.98% area; Density = 602.97
South: 94.23% area; Density = 602.10
North: 93.27% area; Density = 605.52

Total: 6.02% area; Density = 1088.71
South: 5.77% area; Density = 1092.41
North: 6.73% area; Density = 1079.53


In [9]:
# New mask

apply_unwise_spike = True
if apply_unwise_spike:
    wise_maskbits = [0, 1, 2, 3, 4, 6, 7]  # all except the HALO bit
else:
    wise_maskbits = [0, 2, 3, 4, 6, 7]  # all except the SPIKE and HALO bits

# Apply the unWISE maskbits
mask_unwise = np.zeros(len(cat), dtype=bool)
rmask_unwise = np.zeros(len(randoms), dtype=bool)
for bit in wise_maskbits:
    mask_unwise |= (cat['WISEMASK_W1'] & 2**bit)>0
    rmask_unwise |= (randoms['WISEMASK_W1'] & 2**bit)>0

wise_mask = cat['wise_mask'] | mask_unwise
rwise_mask = randoms['wise_mask'] | rmask_unwise

new_mask = wise_mask | cat['gaia_mask']
rnew_mask = rwise_mask | randoms['gaia_mask']
# new_mask = wise_mask | cat['gaia_bright_mask']
# rnew_mask = rwise_mask | randoms['gaia_bright_mask']
mask_clean = ~(new_mask)
rmask_clean = ~(rnew_mask)

# Unmasked objects
print_summary(mask_clean, rmask_clean)
print_summary(mask_clean, rmask_clean, field='south')
print_summary(mask_clean, rmask_clean, field='north')
print()

# Masked objects
print_summary(new_mask, rnew_mask)
print_summary(new_mask, rnew_mask, field='south')
print_summary(new_mask, rnew_mask, field='north')

Total: 92.21% area; Density = 601.10
South: 92.87% area; Density = 599.54
North: 90.30% area; Density = 605.75

Total: 7.79% area; Density = 1000.09
South: 7.13% area; Density = 1032.00
North: 9.70% area; Density = 932.18


In [10]:
# In new mask but not in DR9 mask
mask = new_mask & (~dr9_mask)
rmask = rnew_mask & (~rdr9_mask)
print_summary(mask, rmask)
print_summary(mask, rmask, field='south')
print_summary(mask, rmask, field='north')

Total: 2.78% area; Density = 664.71
South: 2.43% area; Density = 698.53
North: 3.81% area; Density = 602.25


In [11]:
# In DR9 mask but not in new mask
mask = dr9_mask & (~new_mask)
rmask = rdr9_mask & (~rnew_mask)

print_summary(mask, rmask)
print_summary(mask, rmask, field='south')
print_summary(mask, rmask, field='north')

Total: 1.01% area; Density = 602.11
South: 1.06% area; Density = 598.31
North: 0.84% area; Density = 616.13


---------
## Same new WISE mask, compare GAIA mask

In [12]:
# New WISE mask + DR9 GAIA mask

dr9_maskbits = [1, 11, 12, 13]

dr9_mask = np.zeros(len(cat), dtype=bool)
rdr9_mask = np.zeros(len(randoms), dtype=bool)
for bit in dr9_maskbits:
    dr9_mask |= (cat['MASKBITS'] & 2**bit)>0
    rdr9_mask |= (randoms['MASKBITS'] & 2**bit)>0
    
dr9_mask |= wise_mask
rdr9_mask |= rwise_mask

dr9_clean = ~dr9_mask
rdr9_clean = ~rdr9_mask

# Unmasked objects
print_summary(dr9_clean, rdr9_clean)
print_summary(dr9_clean, rdr9_clean, field='south')
print_summary(dr9_clean, rdr9_clean, field='north')
print()

# Masked objects
print_summary(dr9_mask, rdr9_mask)
print_summary(dr9_mask, rdr9_mask, field='south')
print_summary(dr9_mask, rdr9_mask, field='north')

Total: 93.40% area; Density = 600.67
South: 93.49% area; Density = 599.65
North: 93.15% area; Density = 603.64

Total: 6.60% area; Density = 1078.31
South: 6.51% area; Density = 1071.66
North: 6.85% area; Density = 1096.60


In [13]:
# In new mask but not in DR9 mask
mask = new_mask & (~dr9_mask)
rmask = rnew_mask & (~rdr9_mask)
print_summary(mask, rmask)
print_summary(mask, rmask, field='south')
print_summary(mask, rmask, field='north')

Total: 1.70% area; Density = 576.69
South: 1.30% area; Density = 606.82
North: 2.86% area; Density = 537.05


In [14]:
# In DR9 mask but not in new mask
mask = dr9_mask & (~new_mask)
rmask = rdr9_mask & (~rnew_mask)

print_summary(mask, rmask)
print_summary(mask, rmask, field='south')
print_summary(mask, rmask, field='north')

Total: 0.50% area; Density = 598.17
South: 0.68% area; Density = 597.91
North: 0.01% area; Density = 679.25


---------
## Same new GAIA mask, compare WISE mask

In [15]:
# New GAIA mask + DR9 WISE mask

dr9_mask = cat['gaia_mask'].copy()
rdr9_mask = randoms['gaia_mask'].copy()
dr9_mask |= cat['MASKBITS'] & 2**8 > 0
rdr9_mask |= randoms['MASKBITS'] & 2**8 > 0

dr9_clean = ~dr9_mask
rdr9_clean = ~rdr9_mask

# Unmasked objects
print_summary(dr9_clean, rdr9_clean)
print_summary(dr9_clean, rdr9_clean, field='south')
print_summary(dr9_clean, rdr9_clean, field='north')
print()

# Masked objects
print_summary(dr9_mask, rdr9_mask)
print_summary(dr9_mask, rdr9_mask, field='south')
print_summary(dr9_mask, rdr9_mask, field='north')

Total: 92.79% area; Density = 603.44
South: 93.61% area; Density = 602.03
North: 90.43% area; Density = 607.64

Total: 7.21% area; Density = 1002.33
South: 6.39% area; Density = 1045.54
North: 9.57% area; Density = 918.80


In [16]:
# In new mask but not in DR9 mask
mask = new_mask & (~dr9_mask)
rmask = rnew_mask & (~rdr9_mask)
print_summary(mask, rmask)
print_summary(mask, rmask, field='south')
print_summary(mask, rmask, field='north')

Total: 1.09% area; Density = 802.06
South: 1.14% area; Density = 805.12
North: 0.96% area; Density = 791.59


In [17]:
# In DR9 mask but not in new mask
mask = dr9_mask & (~new_mask)
rmask = rdr9_mask & (~rnew_mask)

print_summary(mask, rmask)
print_summary(mask, rmask, field='south')
print_summary(mask, rmask, field='north')

Total: 0.51% area; Density = 606.56
South: 0.40% area; Density = 600.03
North: 0.83% area; Density = 615.60
